# Applio One Click Gradio

กดรันเซลล์เดียวด้านล่าง แล้วรอจนขึ้นลิงก์ Gradio

In [ ]:
# @title กดรันครั้งเดียวเพื่อเปิด Applio Gradio
# @markdown ถ้า Colab ขอสิทธิ์ Google Drive ให้กดยืนยัน 1 ครั้ง แล้วปล่อยให้รันต่อ
mount_drive = True # @param {type:"boolean"}
sync_models_with_drive = True # @param {type:"boolean"}
sharing_method = "gradio" # @param ["gradio", "localtunnel", "ngrok"]
ngrok_token = "" # @param {type:"string"}
tensorboard = False # @param {type:"boolean"}
install_medleyvox = True # @param {type:"boolean"}
# @markdown install_medleyvox = ติดตั้ง MedleyVox (แยกเสียงนักร้อง 2 คน) เพิ่มแท็บใน Plugins ใช้เวลาเพิ่มประมาณ 10 นาที ปิดได้ถ้าไม่ใช้
language = "th_TH" # @param ["th_TH", "en_US"]

from IPython.display import clear_output
from pathlib import Path
import os

repo_url = "https://github.com/IAHispano/Applio/"
repo_name = "Applio"
repo_path = Path('/content') / repo_name
logs_path = repo_path / 'logs'
backups_path = Path('/content/drive/MyDrive/ApplioBackup')

if mount_drive:
    from google.colab import drive
    drive.mount('/content/drive')

%cd /content
!git config --global advice.detachedHead false
if repo_path.exists():
    %cd /content/Applio
    !git checkout -- tabs/inference/inference.py
    !git pull
else:
    !git clone {repo_url}
    %cd /content/Applio

clear_output()
print('กำลังติดตั้งของจำเป็น รอสักครู่...')
!apt update -y
!apt install -y portaudio19-dev
!curl -LsSf https://astral.sh/uv/install.sh | sh
!uv pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match
!uv pip install -q ngrok jupyter-ui-poll
!npm install -g -q localtunnel > /dev/null 2>&1
!python core.py prerequisites --models --pretraineds-hifigan

if mount_drive and sync_models_with_drive and Path('/content/drive').is_mount():
    print('กำลังเชื่อมโฟลเดอร์โมเดลกับ Google Drive...')
    backups_path.mkdir(parents=True, exist_ok=True)
    logs_path.mkdir(parents=True, exist_ok=True)
    temp_names = {'mute', 'reference', 'zips', 'mute_spin', 'mute_spin-v2'}
    for item in list(logs_path.iterdir()):
        if item.name not in temp_names and item.is_dir() and not item.is_symlink():
            target = backups_path / item.name
            if not target.exists():
                !mv "{item}" "{target}"
    for item in list(backups_path.iterdir()):
        link = logs_path / item.name
        if item.name not in temp_names and not link.exists():
            !ln -s "{item}" "{link}"

for name in ['mute', 'reference', 'zips', 'mute_spin', 'mute_spin-v2']:
    path = logs_path / name
    if path.exists() and (path.is_symlink() or not path.is_dir()):
        !rm -rf "{path}"
    path.mkdir(parents=True, exist_ok=True)

clear_output()
print('ติดตั้งเสร็จแล้ว กำลังเปิด Gradio...')

# ตั้งภาษาหน้าเว็บ (Applio มีภาษาไทยในตัว: assets/i18n/languages/th_TH.json)
import json
os.environ['LANG'] = f'{language}.UTF-8'
config_path = repo_path / 'assets' / 'config.json'
template_path = repo_path / 'assets' / 'config_template.json'
if not config_path.exists() and template_path.exists():
    import shutil
    shutil.copy(template_path, config_path)  # รันครั้งแรกยังไม่มีไฟล์ตั้งค่า สร้างจากต้นแบบก่อน
if config_path.exists():
    config = json.loads(config_path.read_text(encoding='utf-8'))
    config.setdefault('lang', {})
    config['lang']['override'] = True
    config['lang']['selected_lang'] = language
    config_path.write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding='utf-8')

# แก้เสียงที่แปลงแล้วเล่นไม่ขึ้น/เล่นเสียงของโมเดลก่อนหน้า:
# Applio บันทึกผลด้วยชื่อเดิมทุกครั้ง เว็บและเบราว์เซอร์เลยเปิดไฟล์เก่าที่จำไว้ -> คัดลอกเป็นชื่อใหม่ทุกครั้ง (ชื่อเพลง_ชื่อโมเดล_เวลา)
inference_py = repo_path / 'tabs' / 'inference' / 'inference.py'
old_code = """                result = run_infer_script(*args)
                gr.Info(result[0])
                return result
"""
new_code = """                result = run_infer_script(*args)
                gr.Info(result[0])
                try:
                    import shutil, time, re as _re
                    src_path = result[1]
                    model = next((a for a in args if isinstance(a, str) and a.endswith('.pth')), '')
                    model = os.path.splitext(os.path.basename(model))[0]
                    stem, ext = os.path.splitext(os.path.basename(src_path))
                    stem = _re.sub(r'_output$', '', stem)
                    name = '_'.join(x for x in [stem, model, time.strftime('%H%M%S')] if x) + ext
                    new_path = os.path.join(os.path.dirname(src_path), name)
                    shutil.copy(src_path, new_path)
                    result = (result[0], new_path)
                except Exception:
                    traceback.print_exc()
                return result
"""
text = inference_py.read_text(encoding='utf-8')
if old_code in text:
    text = text.replace(old_code, new_code, 1)
    print('แก้ปัญหาเสียงเล่นไฟล์เก่าแล้ว')
else:
    print('คำเตือน: Applio เปลี่ยนโค้ด หาจุดแก้เสียงเล่นไฟล์เก่าไม่เจอ (ใช้งานต่อได้ตามปกติ)')

# รูปคลื่นขึ้นช้า: ส่ง MP3 ตัวอย่างไฟล์เล็กไปวาดรูปคลื่น/ฟัง (เร็วขึ้นประมาณ 10 เท่า) + เพิ่มช่องโหลดไฟล์เต็ม
NL = chr(10)
fast_patch = {
    'from rvc.lib.utils import format_title' + NL: NL.join([
        'from rvc.lib.utils import format_title',
        '',
        '',
        'def make_preview(path):',
        '    try:',
        '        import librosa',
        '        import soundfile as sf',
        '        preview_dir = os.path.join(now_dir, "assets", "audios", "_preview")',
        '        os.makedirs(preview_dir, exist_ok=True)',
        '        out = os.path.join(preview_dir, os.path.splitext(os.path.basename(path))[0] + ".mp3")',
        '        audio, sr = librosa.load(path, sr=None, mono=True)',
        '        sf.write(out, audio, sr, format="mp3")',
        '        return out',
        '    except Exception:',
        '        traceback.print_exc()',
        '        return path',
        '']),
    '        gr.Markdown(value=i18n("## Conversion"))' + NL: NL.join([
        '        def enforce_terms_fast(*a):',
        '            r = enforce_terms(*a)',
        '            if not isinstance(r, tuple):',
        '                r = (r, None)',
        '            msg, path = r[0], r[1]',
        '            return msg, (make_preview(path) if path else None), path',
        '',
        '        gr.Markdown(value=i18n("## Conversion"))',
        '']),
    '            vc_output2 = gr.Audio(label=i18n("Export Audio"))' + NL: NL.join([
        '            vc_output2 = gr.Audio(label=i18n("Export Audio"))',
        '            vc_output_full = gr.File(label="โหลดไฟล์เต็ม (คุณภาพเต็ม)")',
        '']),
    '        fn=enforce_terms,' + NL: '        fn=enforce_terms_fast,' + NL,
    '        outputs=[vc_output1, vc_output2],' + NL: '        outputs=[vc_output1, vc_output2, vc_output_full],' + NL,
}
if all(k in text for k in fast_patch):
    for k, v in fast_patch.items():
        text = text.replace(k, v, 1)
    print('แก้รูปคลื่นขึ้นช้าแล้ว')
else:
    print('คำเตือน: Applio เปลี่ยนโค้ด หาจุดแก้รูปคลื่นขึ้นช้าไม่เจอ (ใช้งานต่อได้ตามปกติ)')
inference_py.write_text(text, encoding='utf-8')

# ---- MedleyVox: แยกเสียงนักร้อง 2 คน (ติดตั้งแยกบน Python 3.10 เพราะ fairseq ใช้กับ Python ของ Colab ไม่ได้) ----
mv_path = Path('/content/MedleyVox-Inference-WebUI')
mv_py = '/content/mvenv/bin/python'
if install_medleyvox:
    import subprocess, sys

    def run(step, cmd):
        # รันทีละขั้น ถ้าพังให้หยุดและแสดงสาเหตุ
        print(f'▶ {step} ...')
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if r.returncode != 0:
            print((r.stdout + r.stderr)[-3000:])
            raise SystemExit(f'❌ ขั้น "{step}" ไม่สำเร็จ — ส่งข้อความด้านบนให้คนดูแลได้เลย')

    if not mv_path.exists():
        run('ดาวน์โหลด MedleyVox', f'git clone -q https://github.com/SUC-DriverOld/MedleyVox-Inference-WebUI {mv_path}')
    if not Path('/content/mvenv/.ok').exists():
        print('กำลังติดตั้ง MedleyVox (ประมาณ 10 นาที) ...')
        run('ติดตั้งตัวช่วย uv', f'{sys.executable} -m pip install -q uv')
        run('สร้าง Python 3.10', f'rm -rf /content/mvenv && {sys.executable} -m uv venv --seed -p 3.10 /content/mvenv')
        run('เตรียม pip', f'{mv_py} -m pip install -q "pip<24.1" setuptools wheel cython numpy')
        run('ติดตั้ง torch', f'{mv_py} -m pip install -q "torch<2.6.0" torchaudio --index-url https://download.pytorch.org/whl/cu124')
        run('ติดตั้งส่วนประกอบ MedleyVox (นานสุด)', f'{mv_py} -m pip install -q -r {mv_path}/requirements.txt --extra-index-url https://download.pytorch.org/whl/cu124')
        Path('/content/mvenv/.ok').touch()
    run('ตรึง setuptools ให้ librosa ใช้ได้', f'{mv_py} -m pip install -q "setuptools<81"')
    run('ตรวจว่า MedleyVox ติดตั้งครบ', f'{mv_py} -c "import pkg_resources, torch, fairseq, asteroid, librosa"')

    print('กำลังดาวน์โหลดโมเดล MedleyVox ...')
    mv_models = {
        'vocals_238': 'vocals%20238',
        'multi_singing_librispeech_138': 'multi_singing_librispeech_138',
        'singing_librispeech_ft_iSRNet': 'singing_librispeech_ft_iSRNet',
    }
    for name, remote in mv_models.items():
        folder = mv_path / 'checkpoints' / name
        folder.mkdir(parents=True, exist_ok=True)
        for f in ['vocals.pth', 'vocals.json']:
            if not (folder / f).exists():
                run(f'โหลดโมเดล {name}/{f}', f'wget -q -O "{folder}/{f}" "https://huggingface.co/Cyru5/MedleyVox/resolve/main/{remote}/{f}"')
    (mv_path / 'pretrained').mkdir(exist_ok=True)
    if not (mv_path / 'pretrained' / 'xlsr_53_56k.pt').exists():
        run('โหลดโมเดล xlsr_53_56k.pt', f'wget -q -O "{mv_path}/pretrained/xlsr_53_56k.pt" https://dl.fbaipublicfiles.com/fairseq/wav2vec/xlsr_53_56k.pt')
    # inference.py ฉบับที่รายงานความคืบหน้า (ใช้ทำแถบ % ในหน้าเว็บ)
    (mv_path / 'inference.py').write_text('import os\nimport sys\nsys.path.append(os.path.dirname(os.path.abspath(__file__)))\nimport json\nimport argparse\nimport numpy as np\nimport soundfile as sf\nimport librosa\nimport torch\nimport pyloudnorm as pyln\nimport tqdm\nimport traceback\n\nfrom models import load_model_with_args\nfrom functions import load_ola_func_with_args\nfrom utils import loudnorm, str2bool, db2linear\n\nimport warnings\nwarnings.filterwarnings("ignore")\n\nmodel = None\ncontinuous_nnet = None\ndevice = torch.device("cpu")\n\n# ---- ความคืบหน้า: จดลงไฟล์ให้หน้าเว็บอ่าน (นับจำนวนท่อนที่โมเดลแยกเสร็จ) ----\nimport time\nprogress = {"stage": "loading", "file": "", "file_idx": 0, "n_files": 0, "done": 0, "total": None, "start": time.time(), "file_start": time.time()}\n_last_write = [0.0]\n\ndef write_progress(force=False):\n    path = os.environ.get("MEDLEYVOX_PROGRESS")\n    if not path or (not force and time.time() - _last_write[0] < 0.5):\n        return\n    _last_write[0] = time.time()\n    try:\n        with open(path + ".tmp", "w", encoding="utf-8") as f:\n            json.dump(progress, f, ensure_ascii=False)\n        os.replace(path + ".tmp", path)\n    except Exception:\n        pass\n\ndef count_chunks(args, n_samples):\n    # จำนวนท่อนที่ต้องแยกต่อ 1 ช่องเสียง (รู้ล่วงหน้าได้เฉพาะ None / ola / ola_norm / w2v)\n    if not args.use_overlapadd:\n        return 1\n    if args.use_overlapadd not in ("ola", "ola_norm", "w2v"):\n        return None\n    win = int(args.ola_window_len * args.sample_rate) if args.ola_window_len else int(args.seq_dur * args.sample_rate)\n    hop = int(args.ola_hop_len * args.sample_rate) if args.ola_hop_len else int(args.seq_dur * args.sample_rate // 4)\n    # เหมือนที่ asteroid หั่น: เติมขอบหัวท้ายข้างละ 1 ท่อน แล้วเลื่อนทีละ hop\n    return (n_samples + win) // hop + 1\n\n\ndef main():\n    global model, continuous_nnet, device\n\n    parser = argparse.ArgumentParser(description="Inference for MedleyVox", formatter_class=lambda prog: argparse.RawTextHelpFormatter(prog, max_help_position=60))\n    parser.add_argument("--target", type=str, default="vocals")\n    parser.add_argument("--exp_name", type=str, default=None)\n    parser.add_argument("--model_dir", type=str, default="checkpoints", help="model directory")\n    parser.add_argument("--use_gpu", type=str2bool, default=True)\n    parser.add_argument("--use_overlapadd", type=str, default=None, choices=[None, "ola", "ola_norm", "w2v", "w2v_chunk", "sf_chunk"], help="use overlapadd functions, ola, ola_norm, w2v will work with ola_window_len, ola_hop_len argugments. w2v_chunk and sf_chunk is chunk-wise processing based on VAD, so you have to specify the vad_method args. If you use sf_chunk (spectral_featrues_chunk), you also need to specify spectral_features.")\n    parser.add_argument("--vad_method", type=str, default="spec", choices=["spec", "webrtc"], help="what method do you want to use for \'voice activity detection (vad) -- split chunks -- processing. Only valid when \'w2v_chunk\' or \'sf_chunk\' for args.use_overlapadd.")\n    parser.add_argument("--spectral_features", type=str, default="mfcc", choices=["mfcc", "spectral_centroid"], help="what spectral feature do you want to use in correlation calc in speaker assignment (only valid when using sf_chunk)")\n    parser.add_argument("--w2v_ckpt_dir", type=str, default="pretrained", help="only valid when use_overlapadd is \'w2v\' or \'w2v_chunk\'.")\n    parser.add_argument("--w2v_nth_layer_output", nargs="+", type=int, default=[0], help="wav2vec nth layer output")\n    parser.add_argument("--ola_window_len", type=float, default=None, help="ola window size in [sec]")\n    parser.add_argument("--ola_hop_len", type=float, default=None, help="ola hop size in [sec]")\n    parser.add_argument("--use_ema_model", type=str2bool, default=True, help="use ema model or online model? only vaind when args.ema it True (model trained with ema)")\n    parser.add_argument("--mix_consistent_out", type=str2bool, default=True, help="only valid when the model is trained with mixture_consistency loss. Default is True.")\n    parser.add_argument("--reorder_chunks", type=str2bool, default=True, help="ola reorder chunks")\n    parser.add_argument("--inference_data_dir", type=str, default="inputs", help="data where you want to separate")\n    parser.add_argument("--results_save_dir", type=str, default="results", help="save directory")\n    parser.add_argument("--output_format", type=str, default="wav", choices=["wav", "mp3", "flac"], help="output format")\n    parser.add_argument("--separate_storage", type=str2bool, default=False, help="save results in separate folders with the same name as the input file")\n    parser.add_argument("--skip_error", type=str2bool, default=True, help="skip error files while separating instead of stopping")\n    args, _ = parser.parse_known_args()\n\n    args.exp_result_dir = os.path.join(args.model_dir, args.exp_name)\n    with open(os.path.join(args.exp_result_dir, f"{args.target}.json"), "r") as f:\n        args_dict = json.load(f)\n    for key, value in args_dict["args"].items():\n        setattr(args, key, value)\n\n    if args.use_overlapadd == "w2v" or args.use_overlapadd == "w2v_chunk":\n        assert os.path.exists(os.path.join(args.w2v_ckpt_dir, "xlsr_53_56k.pt")), "Please download the pretrained model xlsr_53_56k.pt from https://dl.fbaipublicfiles.com/fairseq/wav2vec/xlsr_53_56k.pt and put it in the \'pretrained\' folder!"\n\n    # load model architecture\n    model = load_model_with_args(args)\n\n    if torch.cuda.is_available() and args.use_gpu:\n        device = torch.device("cuda")\n        print("Using GPU for inference")\n    else:\n        print("Using CPU for inference")\n\n    target_model_path = f"{args.exp_result_dir}/{args.target}.pth"\n    checkpoint = torch.load(target_model_path, map_location=device)\n    if args.ema and args.use_ema_model:\n        print("Use ema model")\n        model_dict = model.state_dict()\n        # 1. filter out unnecessary keys\n        checkpoint = {\n            k.replace("ema_model.module.", ""): v\n            for k, v in checkpoint.items()\n            if k.replace("ema_model.module.", "") in model_dict\n        }\n        # 2. overwrite entries in the existing state dict\n        model_dict.update(checkpoint)\n        # 3. load the new state dict\n        model.load_state_dict(model_dict)\n    elif args.ema and not args.use_ema_model:\n        print("Use ema online model")\n        model_dict = model.state_dict()\n        # 1. filter out unnecessary keys\n        checkpoint = {\n            k.replace("online_model.module.", ""): v\n            for k, v in checkpoint.items()\n            if k.replace("online_model.module.", "") in model_dict\n        }\n        # 2. overwrite entries in the existing state dict\n        model_dict.update(checkpoint)\n        # 3. load the new state dict\n        model.load_state_dict(model_dict)\n    else:\n        model.load_state_dict(checkpoint)\n\n    model.eval()\n    meter = pyln.Meter(args.sample_rate)\n\n    _forward = model.forward\n    def counted_forward(*a, **k):\n        out = _forward(*a, **k)\n        progress["done"] += 1\n        write_progress()\n        return out\n    model.forward = counted_forward\n\n    if args.use_overlapadd:\n        continuous_nnet = load_ola_func_with_args(args, model, device, meter)\n\n    data_list = os.listdir(args.inference_data_dir)\n    print(f"Total files found: {len(data_list)}")\n    progress["n_files"] = len(data_list)\n    data_list = tqdm.tqdm(data_list, desc="Separating")\n\n    success_files = []\n    error_files = []\n    for idx, data_path in enumerate(data_list):\n        song_name = os.path.basename(data_path).split(".")[0]\n        progress.update(stage="reading", file=data_path, file_idx=idx + 1, done=0, total=None, file_start=time.time())\n        write_progress(force=True)\n        data_list.set_postfix_str(song_name)\n\n        if args.separate_storage:\n            save_dir = os.path.join(args.results_save_dir, song_name)\n        else:\n            save_dir = args.results_save_dir\n        os.makedirs(save_dir, exist_ok=True)\n\n        try:\n            mixture, _ = librosa.load(\n                path=os.path.join(args.inference_data_dir, data_path),\n                sr=args.sample_rate,\n                mono=False,\n                dtype=np.float32\n            )\n        except Exception as e:\n            print(f"Error loading {data_path}, error: {e}, skipping...")\n            traceback.print_exc()\n            error_files.append(data_path)\n            continue\n\n        try:\n            if len(mixture.shape) != 1 and mixture.shape[0] > 2:\n                mixture = np.mean(mixture, axis=0)\n\n            per_channel = count_chunks(args, mixture.shape[-1])\n            channels = 2 if len(mixture.shape) == 2 else 1\n            progress.update(stage="separating", done=0, total=per_channel * channels if per_channel else None, file_start=time.time())\n            write_progress(force=True)\n\n            if len(mixture.shape) == 2:\n                left = mixture[0, :]\n                right = mixture[1, :]\n                left_out_wav_1, left_out_wav_2 = inference(args, left, meter)\n                right_out_wav_1, right_out_wav_2 = inference(args, right, meter)\n                out_wav_1 = np.stack([left_out_wav_1, right_out_wav_1], axis=-1)\n                out_wav_2 = np.stack([left_out_wav_2, right_out_wav_2], axis=-1)\n            else:\n                out_wav_1, out_wav_2 = inference(args, mixture, meter)\n\n            save_wav_path_1 = os.path.join(save_dir, f"{song_name}_output_1.{args.output_format}")\n            save_wav_path_2 = os.path.join(save_dir, f"{song_name}_output_2.{args.output_format}")\n\n            progress["stage"] = "saving"\n            write_progress(force=True)\n            sf.write(save_wav_path_1, out_wav_1, args.sample_rate)\n            sf.write(save_wav_path_2, out_wav_2, args.sample_rate)\n            success_files.append(data_path)\n        except Exception as e:\n            if args.skip_error:\n                print(f"Error separating {data_path}, error: {e}, skipping...")\n                traceback.print_exc()\n                error_files.append(data_path)\n                continue\n            else:\n                raise e\n\n    progress.update(stage="finished", success=len(success_files), errors=error_files)\n    write_progress(force=True)\n    print("Separation done!")\n    if len(success_files) == 0:\n        print("No files separated")\n    else:\n        print(f"Successfully separated {len(success_files)} files: {success_files}")\n    if len(error_files) == 0:\n        print("No files with errors")\n    else:\n        print(f"Error in separating {len(error_files)} files: {error_files}")\n\ndef inference(args, mix, meter):\n    assert (model is not None or continuous_nnet is not None), "model is not loaded"\n\n    mix, adjusted_gain = loudnorm(mix, -24.0, meter)\n    mix = np.expand_dims(mix, axis=0)\n    mix = mix.reshape(1, mix.shape[0], mix.shape[1])\n    mix = torch.as_tensor(mix, dtype=torch.float32).to(device)\n\n    if args.use_overlapadd:\n        out_wavs = continuous_nnet.forward(mix)\n    else:\n        out_wavs = model.separate(mix)\n\n    if args.use_gpu:\n        out_wav_1 = out_wavs[0, 0, :].cpu().detach().numpy()\n        out_wav_2 = out_wavs[0, 1, :].cpu().detach().numpy()\n    else:\n        out_wav_1 = out_wavs[0, 0, :]\n        out_wav_2 = out_wavs[0, 1, :]\n\n    out_wav_1 = out_wav_1 * db2linear(-adjusted_gain)\n    out_wav_2 = out_wav_2 * db2linear(-adjusted_gain)\n\n    return out_wav_1, out_wav_2\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
    print('MedleyVox พร้อมแล้ว')

# ปลั๊กอิน (แท็บ Plugins): MultiModelDownload = โหลดโมเดลหลายตัวจากลิงก์ / MultiModelConvert = แปลงเพลงเดียวด้วยหลายโมเดล / MedleyVoxSeparate = แยกเสียงนักร้อง 2 คน
plugin_sources = {'MultiModelDownload': '# ปลั๊กอิน Applio: วางลิงก์ Hugging Face แล้วเลือกโหลดโมเดลหลายตัวพร้อมกัน\nimport os\nimport re\nimport shutil\nimport requests\nimport gradio as gr\n\nnow_dir = os.getcwd()\nlogs_dir = os.path.join(now_dir, "logs")\nzips_dir = os.path.join(logs_dir, "zips")\ndrive_backup = "/content/drive/MyDrive/ApplioBackup"\n\n\ndef parse_link(link):\n    m = re.match(r"https?://huggingface\\.co/([^/\\s]+/[^/\\s]+)(?:/(?:tree|blob|resolve)/([^/\\s]+)/?(.*))?", link.strip())\n    if not m:\n        raise gr.Error("ลิงก์ไม่ถูกต้อง ต้องเป็นลิงก์ huggingface.co เช่น https://huggingface.co/ชื่อ/ชื่อโมเดล/tree/main")\n    return m.group(1), m.group(2) or "main", (m.group(3) or "").strip("/")\n\n\ndef source_label(url):\n    # ที่มาแบบสั้น: Hugging Face = เจ้าของ/ชื่อ repo ; Google Drive ; อื่น ๆ = ชื่อเว็บ\n    m = re.match(r"https?://huggingface\\.co/([^/\\s]+/[^/\\s?#]+)", url or "")\n    if m: return m.group(1)\n    if "drive.google.com" in (url or ""): return "Google Drive"\n    m = re.match(r"https?://([^/\\s]+)", url or "")\n    return m.group(1) if m else ""\n\n\ndef save_source(folder, url):\n    # 2026-10-03 ผู้ใช้ "จำไม่ได้ว่าโมเดลมาจากลิงก์ไหน": จดที่มาไว้ในโฟลเดอร์โมเดล -> หน้าแปลงหลายโมเดลโชว์ต่อท้ายชื่อ\n    try:\n        import json\n        if os.path.isdir(folder):\n            with open(os.path.join(folder, "_source.json"), "w", encoding="utf-8") as f:\n                json.dump({"url": url, "source": source_label(url)}, f, ensure_ascii=False)\n    except Exception:\n        pass\n\n\ndef is_folder_link(link):\n    # หน้าโฟลเดอร์ของ Hugging Face (ไม่ได้ชี้ไปที่ไฟล์) -> แสดงรายการให้ติ๊ก ; อย่างอื่น = โหลดทันที\n    m = re.match(r"https?://huggingface\\.co/[^/\\s]+/[^/\\s]+(?:/tree/[^/\\s]+(/.*)?)?/?$", link.strip())\n    return bool(m) and not re.search(r"\\.(zip|pth|index)(\\?.*)?$", link.strip(), re.I)\n\n\ndef snapshot_models():\n    return set(os.listdir(logs_dir)) if os.path.isdir(logs_dir) else set()\n\n\ndef download_direct(links, progress):\n    # 2026-10-03 ผู้ใช้: "มันโหลดโมเดลเดียวไม่ได้ ขี้เกียจสลับหน้า" -> ลิงก์ไฟล์เดียว/Google Drive/หลายลิงก์ ใช้ตัวโหลดของ Applio เอง\n    from rvc.lib.tools.model_download import model_download_pipeline\n    report = []\n    for i, url in enumerate(links):\n        progress(i / len(links), desc=f"กำลังโหลด ({i + 1}/{len(links)})")\n        before = snapshot_models()\n        try:\n            result = model_download_pipeline(url)\n            if result == "Error":\n                raise RuntimeError("โหลดหรือแตกไฟล์ไม่สำเร็จ")\n            new = sorted(snapshot_models() - before - {"zips", "mute"})\n            for name in new:\n                save_source(os.path.join(logs_dir, name), url)\n                keep_folder_in_drive(os.path.join(logs_dir, name))\n            report.append(f"✅ {\', \'.join(new) or url}")\n        except Exception as e:\n            report.append(f"❌ {url} : {e}")\n    progress(1, desc="เสร็จแล้ว")\n    ok = sum(r.startswith("✅") for r in report)\n    return f"โหลดสำเร็จ {ok} จาก {len(links)} ลิงก์\\n" + "\\n".join(report) + \\\n        "\\n\\nไปที่แท็บแปลงเสียง แล้วกดปุ่มรีเฟรชรายการโมเดล จะเห็นโมเดลใหม่"\n\n\ndef open_link(text, progress=gr.Progress()):\n    links = [l.strip() for l in (text or "").splitlines() if l.strip()]\n    if not links:\n        raise gr.Error("กรุณาวางลิงก์ก่อน")\n    if len(links) == 1 and is_folder_link(links[0]):\n        return list_files(links[0])\n    return gr.update(choices=[], value=[], visible=False), {}, download_direct(links, progress)\n\n\ndef pretty_name(path):\n    # 2026-10-03 ผู้ใช้ "รายชื่อโมเดลดูรก": ตัดที่อยู่โฟลเดอร์ / นามสกุล / ตัวเลขรอบเทรน (e300, 300e, s12000, epoch300) ; เก็บเลขอื่น เช่น 2026, v2\n    name = re.sub(r"\\.(zip|pth|index)$", "", os.path.basename(path), flags=re.I)\n    name = re.sub(r"(?i)(?:^|[_\\-\\s])(?:e|ep|epoch|epochs)\\d+(?=$|[_\\-\\s])|(?:^|[_\\-\\s])\\d+(?:e|ep|epoch|epochs)(?=$|[_\\-\\s])|(?:^|[_\\-\\s])s\\d{3,}(?=$|[_\\-\\s])|(?:^|[_\\-\\s])\\d{3,}s(?:teps?)?(?=$|[_\\-\\s])", " ", name)\n    name = re.sub(r"[_\\s]+", " ", name).strip(" -_")\n    return name or os.path.basename(path)\n\n\ndef group_of(name):\n    # กลุ่ม = คำแรกของชื่อ (ส่วนใหญ่เป็นชื่อคน เช่น ASA, RUKA)\n    m = re.match(r"[A-Za-z\\u0E00-\\u0E7F]+", name)\n    return m.group(0).upper() if m else "อื่น ๆ"\n\n\ndef nice_choices(items):\n    # items = [(ชื่อเดิม/ค่า, ชื่อที่โชว์, ข้อความท้าย)] -> เรียงตามกลุ่มแล้วชื่อ ; โชว์ "กลุ่ม · ชื่อ   ข้อความท้าย"\n    rows = sorted(items, key=lambda r: (group_of(r[1]), r[1].lower()))\n    def label(r):\n        g = group_of(r[1]); rest = r[1][len(g):].strip(" -_") if r[1].upper().startswith(g) else r[1]\n        return (f"{g} · {rest}" if rest else g) + (f"   ({r[2]})" if r[2] else "")   # "ASA · SOFT v2" ไม่ซ้ำชื่อคน\n    return [(label(r), r[0]) for r in rows]\n\n\ndef list_files(link):\n    repo, rev, sub = parse_link(link)\n    url = f"https://huggingface.co/api/models/{repo}/tree/{rev}" + (f"/{sub}" if sub else "") + "?recursive=1"\n    r = requests.get(url, timeout=30)\n    if r.status_code != 200:\n        raise gr.Error(f"เปิดลิงก์ไม่ได้ (รหัส {r.status_code}) ตรวจว่าลิงก์ถูกและเป็นแบบสาธารณะ")\n    files = [f for f in r.json() if f.get("type") == "file" and f["path"].lower().endswith(".zip")]\n    if not files:\n        raise gr.Error("ไม่พบไฟล์ .zip ในลิงก์นี้")\n    items = [(f["path"], pretty_name(f["path"]), f"{f.get(\'size\', 0) / 1024 / 1024:.0f} MB") for f in files]\n    choices = nice_choices(items)\n    return (\n        gr.update(choices=choices, value=[], visible=True),\n        {"repo": repo, "rev": rev, "_all": [c[1] for c in choices], "_choices": choices},\n        f"พบ {len(files)} โมเดล ติ๊กเลือกตัวที่ต้องการ แล้วกด \\"ดาวน์โหลดที่เลือก\\" (พิมพ์ในช่องค้นหาเพื่อกรองได้)",\n    )\n\n\ndef download_selected(selected, source, progress=gr.Progress()):\n    from rvc.lib.tools.model_download import handle_extraction_process\n\n    if not source:\n        raise gr.Error("กรุณากด \\"แสดงรายการโมเดล\\" ก่อน")\n    if not selected:\n        raise gr.Error("กรุณาติ๊กเลือกอย่างน้อย 1 โมเดล")\n    os.makedirs(zips_dir, exist_ok=True)\n    report = []\n    for i, path in enumerate(selected):\n        name = os.path.basename(path)\n        progress(i / len(selected), desc=f"กำลังโหลด {name} ({i + 1}/{len(selected)})")\n        try:\n            url = f"https://huggingface.co/{source[\'repo\']}/resolve/{source[\'rev\']}/{path}"\n            with requests.get(url, stream=True, timeout=60) as r:\n                r.raise_for_status()\n                with open(os.path.join(zips_dir, name), "wb") as f:\n                    for chunk in r.iter_content(1024 * 1024):\n                        f.write(chunk)\n            result = handle_extraction_process()\n            if result == "Error":\n                raise RuntimeError("แตกไฟล์ zip ไม่สำเร็จ")\n            from rvc.lib.utils import format_title\n            save_source(os.path.join(logs_dir, format_title(os.path.splitext(name)[0])), f"https://huggingface.co/{source[\'repo\']}/blob/{source[\'rev\']}/{path}")\n            keep_in_drive(os.path.splitext(name)[0])\n            report.append(f"✅ {name}")\n        except Exception as e:\n            report.append(f"❌ {name} : {e}")\n    progress(1, desc="เสร็จแล้ว")\n    ok = sum(r.startswith("✅") for r in report)\n    return f"โหลดสำเร็จ {ok} จาก {len(selected)} โมเดล\\n" + "\\n".join(report) + \\\n        "\\n\\nไปที่แท็บแปลงเสียง แล้วกดปุ่มรีเฟรชรายการโมเดล จะเห็นโมเดลใหม่"\n\n\ndef keep_in_drive(zip_stem):\n    # ถ้าเชื่อม Google Drive ไว้ ย้ายโมเดลไปเก็บใน Drive แล้วทำทางลัดไว้ที่เดิม (รอบหน้าไม่ต้องโหลดใหม่)\n    if not os.path.isdir(drive_backup):\n        return\n    from rvc.lib.utils import format_title\n    keep_folder_in_drive(os.path.join(logs_dir, format_title(zip_stem)))\n\n\ndef keep_folder_in_drive(folder):\n    if not os.path.isdir(drive_backup):\n        return\n    target = os.path.join(drive_backup, os.path.basename(folder))\n    if os.path.isdir(folder) and not os.path.islink(folder) and not os.path.exists(target):\n        shutil.move(folder, target)\n        os.symlink(target, folder)\n\n\ndef applio_plugin():\n    gr.Markdown("## โหลดโมเดล (ตัวเดียว หรือหลายตัวพร้อมกัน)\\n"\n                "- หน้าโฟลเดอร์ Hugging Face ที่มีหลาย zip → แสดงรายการให้ติ๊ก\\n"\n                "- ลิงก์ไฟล์เดียว (Hugging Face / Google Drive / ลิงก์ตรง) → โหลดทันที\\n"\n                "- หลายลิงก์ → วางบรรทัดละ 1 ลิงก์")\n    source = gr.State({})\n    with gr.Row():\n        link = gr.Textbox(label="ลิงก์โมเดล (วางได้หลายบรรทัด)", lines=2, placeholder="https://huggingface.co/ชื่อ/ชื่อโมเดล/tree/main\\nhttps://huggingface.co/ชื่อ/ชื่อโมเดล/resolve/main/ASA.zip", scale=4)\n        list_btn = gr.Button("โหลด / แสดงรายการ", variant="primary", scale=1)\n    search = gr.Textbox(label="ค้นหาในรายการ", placeholder="เช่น ASA", visible=True)\n    # รายชื่อโมเดล: แถวละ 1 ตัว เรียงลงมา (เดิมเรียงต่อกันเป็นพรืด อ่านยาก)\n    gr.HTML("<style>.mm-list .wrap{display:grid!important;grid-template-columns:repeat(auto-fill,minmax(260px,1fr));gap:6px!important}.mm-list .wrap label{margin:0!important;width:100%}</style>")\n    picks = gr.CheckboxGroup(label="เลือกโมเดลที่จะโหลด (เลือกได้หลายอัน)", choices=[], visible=False, elem_classes="mm-list")\n    with gr.Row():\n        all_btn = gr.Button("เลือกทั้งหมด")\n        none_btn = gr.Button("ล้างที่เลือก")\n        dl_btn = gr.Button("ดาวน์โหลดที่เลือก", variant="primary")\n    status = gr.Textbox(label="สถานะ", lines=8, interactive=False)\n\n    list_btn.click(open_link, [link], [picks, source, status])\n    def filter_list(q, src, cur):\n        # ตัวที่ติ๊กไว้ยังอยู่เสมอ แม้ค้นหาคำอื่น\n        ch = src.get("_choices", []); cur = cur or []\n        shown = [c for c in ch if c[1] in cur or (q or "").lower() in c[0].lower()]\n        return gr.update(choices=shown, value=cur)\n    search.change(filter_list, [search, source, picks], [picks])\n    all_btn.click(lambda q, s, cur: gr.update(value=list(dict.fromkeys((cur or []) + [c[1] for c in s.get("_choices", []) if (q or "").lower() in c[0].lower()]))), [search, source, picks], [picks])\n    none_btn.click(lambda: gr.update(value=[]), None, [picks])\n    dl_btn.click(download_selected, [picks, source], [status])\n', 'MultiModelConvert': '# ปลั๊กอิน Applio: แปลงเพลงเดียวด้วยหลายโมเดล (ทำทีละโมเดล) แล้วส่งออกรวมเป็น zip\nimport os\nimport re\nimport shutil\nimport zipfile\nimport time\nimport gradio as gr\n\nnow_dir = os.getcwd()\nlogs_dir = os.path.join(now_dir, "logs")\nout_root = os.path.join(now_dir, "assets", "audios", "multi_model_output")\ndrive_out = "/content/drive/MyDrive/ApplioOutput"\nMAX_PLAYERS = 12  # ตัวเล่นเสียงสูงสุดต่อรอบ (เกินนี้โหลดจาก zip)\nPREVIEW_DIR = os.path.join(now_dir, "assets", "audios", "_preview")\n\n\ndef make_preview(path):\n    # ทำ MP3 ตัวอย่างไฟล์เล็ก (เล็กกว่า WAV ประมาณ 10 เท่า) ไว้วาดรูปคลื่น/ฟังในหน้าเว็บ ไฟล์จริงไม่แตะ\n    try:\n        import librosa\n        import soundfile as sf\n        os.makedirs(PREVIEW_DIR, exist_ok=True)\n        out = os.path.join(PREVIEW_DIR, os.path.splitext(os.path.basename(path))[0] + ".mp3")\n        if not os.path.exists(out) or os.path.getmtime(out) < os.path.getmtime(path):\n            audio, sr = librosa.load(path, sr=None, mono=True)\n            sf.write(out, audio, sr, format="mp3")\n        return out\n    except Exception:\n        return path  # ทำไม่ได้ก็ใช้ไฟล์จริงตามเดิม\n\n\ndef on_upload(path):\n    # เก็บไฟล์จริงไว้ใช้แปลง ส่วนที่โชว์ในช่องเป็นตัวอย่างเล็กๆ จะได้ขึ้นเร็ว\n    if not path:\n        return None, gr.update()\n    return path, gr.update(value=make_preview(path))\nskip_folders = {"zips", "mute", "reference", "mute_spin", "mute_spin-v2"}\n\n\ndef clean_name(text):\n    # ชื่อไฟล์อ่านง่าย: เก็บตัวอักษร/ตัวเลข (รวมภาษาไทย) - _ ; ที่เหลือเปลี่ยนเป็น -\n    text = re.sub(r"[^\\w\\-\u0e00-\u0e7f]+", "-", text.strip())\n    return re.sub(r"-{2,}", "-", text).strip("-_") or "audio"\n\n\ndef find_models():\n    models = {}\n    for root, dirs, files in os.walk(logs_dir, followlinks=True):\n        dirs[:] = [d for d in dirs if d not in skip_folders]\n        for f in files:\n            if not f.endswith(".pth") or f.startswith(("G_", "D_")):\n                continue\n            indexes = sorted(\n                (i for i in os.listdir(root) if i.endswith(".index")),\n                key=lambda i: ("trained" in i, i),  # ใช้ไฟล์ added ก่อน trained\n            )\n            name = os.path.splitext(f)[0]\n            if name in models:\n                name = f"{os.path.basename(root)}-{name}"\n            models[name] = (os.path.join(root, f), os.path.join(root, indexes[0]) if indexes else "")\n    return dict(sorted(models.items(), key=lambda kv: kv[0].lower()))\n\n\ndef pretty_name(path):\n    # 2026-10-03 ผู้ใช้ "รายชื่อโมเดลดูรก": ตัดที่อยู่โฟลเดอร์ / นามสกุล / ตัวเลขรอบเทรน (e300, 300e, s12000, epoch300) ; เก็บเลขอื่น เช่น 2026, v2\n    name = re.sub(r"\\.(zip|pth|index)$", "", os.path.basename(path), flags=re.I)\n    name = re.sub(r"(?i)(?:^|[_\\-\\s])(?:e|ep|epoch|epochs)\\d+(?=$|[_\\-\\s])|(?:^|[_\\-\\s])\\d+(?:e|ep|epoch|epochs)(?=$|[_\\-\\s])|(?:^|[_\\-\\s])s\\d{3,}(?=$|[_\\-\\s])|(?:^|[_\\-\\s])\\d{3,}s(?:teps?)?(?=$|[_\\-\\s])", " ", name)\n    name = re.sub(r"[_\\s]+", " ", name).strip(" -_")\n    return name or os.path.basename(path)\n\n\ndef group_of(name):\n    # กลุ่ม = คำแรกของชื่อ (ส่วนใหญ่เป็นชื่อคน เช่น ASA, RUKA)\n    m = re.match(r"[A-Za-z\\u0E00-\\u0E7F]+", name)\n    return m.group(0).upper() if m else "อื่น ๆ"\n\n\ndef nice_choices(items):\n    # items = [(ชื่อเดิม/ค่า, ชื่อที่โชว์, ข้อความท้าย)] -> เรียงตามกลุ่มแล้วชื่อ ; โชว์ "กลุ่ม · ชื่อ   ข้อความท้าย"\n    rows = sorted(items, key=lambda r: (group_of(r[1]), r[1].lower()))\n    def label(r):\n        g = group_of(r[1]); rest = r[1][len(g):].strip(" -_") if r[1].upper().startswith(g) else r[1]\n        return (f"{g} · {rest}" if rest else g) + (f"   ({r[2]})" if r[2] else "")   # "ASA · SOFT v2" ไม่ซ้ำชื่อคน\n    return [(label(r), r[0]) for r in rows]\n\n\ndef unique_path(path):\n    base, ext = os.path.splitext(path)\n    n = 2\n    while os.path.exists(path):\n        path = f"{base}({n}){ext}"\n        n += 1\n    return path\n\n\ndef refresh(search, selected):\n    # ตัวที่ติ๊กไว้แล้วยังแสดงอยู่เสมอ แม้ค้นหาคำอื่น จะได้ไม่หลุดจากที่เลือก\n    selected = [n for n in (selected or []) if n in find_models()]\n    q = search.lower()\n    names = [n for n in find_models() if q in n.lower() or q in pretty_name(n).lower() or q in model_source(n).lower()]\n    shown = [c for c in model_choices() if c[1] in selected or c[1] in names]\n    return gr.update(choices=shown, value=selected), f"พบ {len(names)} โมเดล · เลือกไว้ {len(selected)}"\n\n\ndef model_source(name):\n    # ที่มาที่จดไว้ตอนโหลด (ปลั๊กอินโหลดโมเดล) ; โมเดลที่โหลดก่อนมีระบบนี้ = ไม่รู้ที่มา\n    try:\n        import json\n        with open(os.path.join(os.path.dirname(find_models()[name][0]), "_source.json"), encoding="utf-8") as f:\n            return json.load(f).get("source") or "ไม่รู้ที่มา"\n    except Exception:\n        return "ไม่รู้ที่มา"\n\n\ndef model_choices():\n    return nice_choices([(n, pretty_name(n), model_source(n)) for n in find_models()])\n\n\ndef audio_seconds(path):\n    try:\n        import soundfile as sf\n        return float(sf.info(path).duration)\n    except Exception:\n        try:\n            import librosa\n            return float(librosa.get_duration(path=path))\n        except Exception:\n            return 180.0\n\n\ndef fmt_left(sec):\n    sec = max(0, int(round(sec)))\n    return f"{sec // 60}:{sec % 60:02d}"\n\n\ndef run_with_progress(fn, kwargs, progress, done_before, total, est, label, disp=None):\n    # 2026-10-03 ผู้ใช้ "แอนิเมชันตอนกำลังทำไม่สมูท อยากได้เวลาถอยหลัง":\n    # แปลงในเธรดแยก แล้วขยับแถบทุก 0.1 วิ ตามเวลาที่คาด (ไม่เกิน 95% ของโมเดลนี้จนกว่าจะเสร็จจริง) + เวลาที่เหลือทั้งหมด\n    import threading\n    box = {}\n    def work():\n        try:\n            box["out"] = fn(**kwargs)\n        except Exception as e:\n            box["err"] = e\n    t = threading.Thread(target=work, daemon=True)\n    start = time.time(); t.start()\n    while t.is_alive():\n        el = time.time() - start\n        part = min(0.95, el / est) if est > 0 else 0.5\n        if el > est:   # ช้ากว่าที่คาด: ค่อย ๆ เข้าใกล้ 95% ไม่หยุดนิ่ง\n            part = 0.95 - 0.25 * (est / el)\n        left = max(est - el, 3) + est * (total - done_before - 1)\n        target = (done_before + part) / total\n        if disp is not None:   # ตอนเดาเวลาผิด แถบไหลไปหาตำแหน่งจริงทีละไม่เกิน 3% (ไม่กระโดด ไม่ถอยหลัง)\n            disp["v"] = max(disp["v"], min(target, disp["v"] + 0.03)); target = disp["v"]\n        progress(target, desc=f"{label} ({done_before + 1}/{total}) · เหลือประมาณ {fmt_left(left)}")\n        t.join(0.1)   # ถี่ 10 ครั้ง/วิ แถบไหลต่อเนื่องเหมือนหน้าแปลงปกติของ Applio\n    if "err" in box:\n        raise box["err"]\n    return box["out"], time.time() - start\n\n\ndef convert_all(audio, original, selected, pitch, f0_method, index_rate, protect, volume_envelope,\n                split_audio, clean_audio, export_format, progress=gr.Progress()):\n    from core import run_infer_script\n\n    if original and os.path.exists(original):\n        audio = original\n    if not audio:\n        raise gr.Error("กรุณาใส่ไฟล์เพลงก่อน")\n    if not selected:\n        raise gr.Error("กรุณาติ๊กเลือกอย่างน้อย 1 โมเดล")\n    models = find_models()\n    song = clean_name(os.path.splitext(os.path.basename(audio))[0])\n    out_dir = os.path.join(out_root, song)\n    os.makedirs(out_dir, exist_ok=True)\n    ext = export_format.lower()\n\n    results, report = [], []\n    # เวลาคาดต่อ 1 โมเดล: เริ่มจากความยาวเพลง (ราว 0.18 วิ ต่อเพลง 1 วิ บน Colab GPU) แล้วใช้เวลาจริงเฉลี่ยของโมเดลที่เสร็จแล้ว\n    est, took, disp = max(15.0, audio_seconds(audio) * 0.18), [], {"v": 0.0}\n    for i, name in enumerate(selected):\n        try:\n            pth, index = models[name]\n            temp_wav = os.path.join(out_dir, f"_temp_{i}.wav")\n            (_, made), sec = run_with_progress(run_infer_script, dict(\n                pitch=int(pitch), index_rate=index_rate, volume_envelope=volume_envelope,\n                protect=protect, f0_method=f0_method, input_path=audio, output_path=temp_wav,\n                pth_path=pth, index_path=index, split_audio=split_audio,\n                f0_autotune=False, f0_autotune_strength=1.0, proposed_pitch=False,\n                proposed_pitch_threshold=155.0, clean_audio=clean_audio, clean_strength=0.5,\n                export_format=export_format, embedder_model="contentvec",\n            ), progress, i, len(selected), est, pretty_name(name), disp)\n            took.append(sec); est = sum(took) / len(took)\n            final = unique_path(os.path.join(out_dir, f"{song}_{clean_name(name)}.{ext}"))\n            shutil.move(made, final)\n            if os.path.exists(temp_wav):\n                os.remove(temp_wav)\n            results.append(final)\n            report.append(f"✅ {os.path.basename(final)}")\n        except Exception as e:\n            report.append(f"❌ {name} : {e}")\n\n    zip_path = None\n    if results:\n        zip_path = unique_path(os.path.join(out_root, f"{song}_{len(results)}โมเดล.zip"))\n        with zipfile.ZipFile(zip_path, "w") as z:\n            for f in results:\n                z.write(f, os.path.basename(f))\n        if os.path.isdir(os.path.dirname(drive_out)):\n            os.makedirs(drive_out, exist_ok=True)\n            shutil.copy(zip_path, drive_out)\n            report.append(f"\\nเก็บสำเนาไว้ใน Google Drive: ApplioOutput/{os.path.basename(zip_path)}")\n    progress(1, desc="เสร็จแล้ว")\n    head = f"แปลงสำเร็จ {len(results)} จาก {len(selected)} โมเดล"\n    # ตัวเล่นเสียงของแต่ละโมเดล (ชื่อโมเดลกำกับ) ให้ฟังเทียบได้เลย\n    players = []\n    for f in results[:MAX_PLAYERS]:\n        label = re.sub(r"\\(\\d+\\)$", "", os.path.splitext(os.path.basename(f))[0].split("_", 1)[-1])\n        players.append(gr.update(value=make_preview(f), label=f"🎧 {label}", visible=True))\n    players += [gr.update(value=None, visible=False)] * (MAX_PLAYERS - len(players))\n    if len(results) > MAX_PLAYERS:\n        head += f" (ฟังในหน้านี้ได้ {MAX_PLAYERS} ตัวแรก ที่เหลือโหลดจาก zip)"\n    return [head + "\\n" + "\\n".join(report), zip_path, results] + players\n\n\ndef applio_plugin():\n    gr.Markdown("## แปลงเพลงเดียวด้วยหลายโมเดล\\nใส่เพลง → ติ๊กโมเดล → กด \\"แปลงทั้งหมด\\" ระบบจะแปลงทีละโมเดล แล้วรวมไฟล์เป็น zip ให้โหลดทีเดียว")\n    audio = gr.Audio(label="เพลงที่จะแปลง (ควรเป็นเสียงร้องที่แยกดนตรีออกแล้ว)", type="filepath")\n    original = gr.State(None)\n    with gr.Row():\n        search = gr.Textbox(label="ค้นหาโมเดล", placeholder="เช่น RAMI", scale=3)\n        refresh_btn = gr.Button("รีเฟรชรายการโมเดล", scale=1)\n    # รายชื่อโมเดล: แถวละ 1 ตัว เรียงลงมา (เดิมเรียงต่อกันเป็นพรืด อ่านยาก)\n    gr.HTML("<style>.mm-list .wrap{display:grid!important;grid-template-columns:repeat(auto-fill,minmax(260px,1fr));gap:6px!important}.mm-list .wrap label{margin:0!important;width:100%}</style>")\n    picks = gr.CheckboxGroup(label="เลือกโมเดล (เลือกได้หลายอัน)", choices=model_choices(), elem_classes="mm-list")\n    found = gr.Markdown(f"พบ {len(find_models())} โมเดล")\n    with gr.Row():\n        all_btn = gr.Button("เลือกทั้งหมดที่แสดง")\n        none_btn = gr.Button("ล้างที่เลือก")\n    with gr.Accordion("ตั้งค่าการแปลง (ใช้กับทุกโมเดล)", open=True):\n        pitch = gr.Slider(-24, 24, value=0, step=1, label="ปรับคีย์ (Pitch)", info="ชายเป็นหญิง ประมาณ +12 / หญิงเป็นชาย ประมาณ -12")\n        f0_method = gr.Radio(["rmvpe", "crepe", "crepe-tiny", "fcpe", "swift"], value="rmvpe", label="วิธีจับโน้ต", info="rmvpe เหมาะกับงานส่วนใหญ่")\n        index_rate = gr.Slider(0, 1, value=0.75, step=0.01, label="ความเหมือนเสียงต้นแบบ (Index rate)")\n        protect = gr.Slider(0, 0.5, value=0.5, step=0.01, label="ป้องกันเสียงลม/พยัญชนะ (Protect)")\n        volume_envelope = gr.Slider(0, 1, value=1, step=0.01, label="ความดังตามต้นฉบับ (Volume envelope)")\n        with gr.Row():\n            split_audio = gr.Checkbox(label="แบ่งเพลงเป็นช่วงก่อนแปลง (เพลงยาว)", value=False)\n            clean_audio = gr.Checkbox(label="ลดเสียงรบกวน", value=False)\n        export_format = gr.Radio(["WAV", "MP3", "FLAC", "OGG", "M4A"], value="WAV", label="ชนิดไฟล์ผลลัพธ์")\n    go = gr.Button("แปลงทั้งหมด", variant="primary")\n    status = gr.Textbox(label="สถานะ", lines=8, interactive=False)\n    gr.Markdown("### ฟังเทียบแต่ละโมเดล")\n    players = []\n    for _ in range(MAX_PLAYERS // 2):\n        with gr.Row():\n            players.append(gr.Audio(label="", visible=False, interactive=False))\n            players.append(gr.Audio(label="", visible=False, interactive=False))\n    zip_file = gr.File(label="โหลดทุกไฟล์ (zip)")\n    files = gr.File(label="หรือโหลดทีละไฟล์", file_count="multiple")\n\n    audio.upload(on_upload, [audio], [original, audio])\n    audio.clear(lambda: None, None, [original])\n    search.change(refresh, [search, picks], [picks, found])\n    refresh_btn.click(refresh, [search, picks], [picks, found])\n    all_btn.click(lambda s, cur: gr.update(value=list(dict.fromkeys((cur or []) + [n for n in find_models() if s.lower() in n.lower()]))), [search, picks], [picks])\n    none_btn.click(lambda: gr.update(value=[]), None, [picks])\n    go.click(convert_all,\n             [audio, original, picks, pitch, f0_method, index_rate, protect, volume_envelope, split_audio, clean_audio, export_format],\n             [status, zip_file, files] + players)\n\n    # ลงทะเบียนช่องรับเพลงไว้ให้แท็บอื่น (เช่น MedleyVoxSeparate) ส่งไฟล์มาใส่ได้\n    import builtins\n    shared = getattr(builtins, "_mmmsp_shared", None)\n    if shared is None:\n        shared = builtins._mmmsp_shared = {}\n    shared["convert_audio"], shared["convert_original"], shared["convert_make_preview"] = audio, original, make_preview\n    if "bind_cross_tab" in shared:\n        shared["bind_cross_tab"]()\n', 'MedleyVoxSeparate': '# ปลั๊กอิน Applio: แยกเสียงนักร้อง 2 คนออกจากกันด้วย MedleyVox แล้วส่งต่อไปหน้าแปลงหลายโมเดลได้\nimport os\nimport re\nimport json\nimport time\nimport shutil\nimport builtins\nimport subprocess\nimport gradio as gr\n\nnow_dir = os.getcwd()\nMV_DIR = os.environ.get("MEDLEYVOX_DIR", "/content/MedleyVox-Inference-WebUI")\nMV_PY = os.environ.get("MEDLEYVOX_PY", "/content/mvenv/bin/python")\nOUT_ROOT = os.path.join(now_dir, "assets", "audios", "medleyvox")\nPREVIEW_DIR = os.path.join(now_dir, "assets", "audios", "_preview")\nPROGRESS_FILE = os.path.join(OUT_ROOT, "_progress.json")\nSTAGE_TEXT = {"loading": "กำลังโหลดโมเดล...", "reading": "กำลังอ่านไฟล์เสียง", "separating": "กำลังแยกเสียง",\n              "saving": "กำลังบันทึกไฟล์", "finished": "กำลังจบงาน"}\nMODEL_HINT = ("💡 แนะนำ: เริ่มที่ vocals_238 (คะแนนแยกเสียงดีที่สุดใน 3 ตัว) | "\n              "singing_librispeech_ft_iSRNet = รุ่นต่อยอด เก็บรายละเอียดเสียงแหลม ลองเมื่อผลฟังอู้อี้ | "\n              "multi_singing_librispeech_138 = ลองเมื่อร้องพร้อมกัน 3 คนขึ้นไปหรือคอรัส | "\n              "ทุกตัวแยกออกมาเป็น 2 เสียง ควรใส่เสียงร้องที่แยกดนตรีออกแล้ว")\nOLA_HINT = ("💡 ola ⭐ ใช้ได้กับแทบทุกเพลง | ola_norm = เพลงมีช่วงเบา-ดังต่างกันมาก | "\n            "w2v = เสียง 2 คนสลับไฟล์กันกลางเพลง (ช้ากว่า) | w2v_chunk / sf_chunk = ลองเมื่อ 3 แบบแรกไม่ได้ผล | "\n            "None = เพลงสั้นเท่านั้น")\n\n# ตัวกลางให้ปลั๊กอินคุยกันข้ามแท็บ (MultiModelConvert ลงทะเบียนช่องรับเพลงไว้ที่นี่)\nshared = getattr(builtins, "_mmmsp_shared", None)\nif shared is None:\n    shared = builtins._mmmsp_shared = {}\n\nCURRENT = {"proc": None, "stopped": False}\n\n\ndef installed():\n    return os.path.isfile(os.path.join(MV_DIR, "inference.py")) and os.path.isfile(MV_PY)\n\n\ndef list_models():\n    folder = os.path.join(MV_DIR, "checkpoints")\n    if not os.path.isdir(folder):\n        return []\n    return sorted(m for m in os.listdir(folder) if os.path.isdir(os.path.join(folder, m)))\n\n\ndef clean_name(text):\n    text = re.sub(r"[^\\w\\-\u0e00-\u0e7f]+", "-", text.strip())\n    return re.sub(r"-{2,}", "-", text).strip("-_") or "audio"\n\n\ndef make_preview(path):\n    # MP3 ตัวอย่างไฟล์เล็กไว้วาดรูปคลื่น/ฟัง (ไฟล์จริงไม่แตะ)\n    try:\n        import librosa\n        import soundfile as sf\n        os.makedirs(PREVIEW_DIR, exist_ok=True)\n        out = os.path.join(PREVIEW_DIR, os.path.splitext(os.path.basename(path))[0] + ".mp3")\n        if not os.path.exists(out) or os.path.getmtime(out) < os.path.getmtime(path):\n            audio, sr = librosa.load(path, sr=None, mono=True)\n            sf.write(out, audio, sr, format="mp3")\n        return out\n    except Exception:\n        return path\n\n\ndef on_upload(path):\n    if not path:\n        return None, gr.update()\n    return path, gr.update(value=make_preview(path))\n\n\ndef fmt_time(sec):\n    sec = int(max(sec, 0))\n    return f"{sec // 60}:{sec % 60:02d}"\n\n\ndef progress_text(p, start):\n    elapsed = time.time() - start\n    if not p:\n        return f"⏳ กำลังโหลดโมเดล...\\nผ่านไป {fmt_time(elapsed)}"\n    stage = p.get("stage", "loading")\n    lines = [f"⏳ {STAGE_TEXT.get(stage, stage)} {p.get(\'file\', \'\')}".rstrip()]\n    total, done = p.get("total"), p.get("done", 0)\n    if stage == "separating" and total:\n        frac = min(done / total, 0.99)\n        filled = int(frac * 20)\n        lines.append(f"{\'█\' * filled}{\'░\' * (20 - filled)} {frac * 100:.0f}%")\n        file_elapsed = time.time() - p.get("file_start", time.time())\n        remain = f" · เหลืออีกประมาณ {fmt_time(file_elapsed / frac - file_elapsed)}" if frac >= 0.03 else " · กำลังคำนวณเวลาที่เหลือ..."\n        lines.append(f"ผ่านไป {fmt_time(elapsed)}{remain}")\n    elif stage == "separating":\n        lines.append(f"แยกเสร็จแล้ว {done} ท่อน · ผ่านไป {fmt_time(elapsed)} (วิธีนี้คำนวณเวลาที่เหลือไม่ได้)")\n    else:\n        lines.append(f"ผ่านไป {fmt_time(elapsed)}")\n    return "\\n".join(lines)\n\n\ndef read_progress():\n    try:\n        with open(PROGRESS_FILE, encoding="utf-8") as f:\n            return json.load(f)\n    except Exception:\n        return None\n\n\ndef hidden_outputs():\n    return [gr.update(value=None, visible=False)] * 2 + [None, None] + [None, None] + [gr.update(visible=False)] * 2\n\n\ndef separate(audio, original, model, overlapadd, output_format):\n    """generator: ส่งข้อความความคืบหน้าเรื่อยๆ แล้วจบด้วยไฟล์ผลลัพธ์"""\n    if original and os.path.exists(original):\n        audio = original\n    if not installed():\n        yield ["❌ ยังไม่ได้ติดตั้ง MedleyVox ในรอบนี้ (เปิดสวิตช์ install_medleyvox ในโน้ตบุ๊กแล้วรันใหม่)"] + hidden_outputs()\n        return\n    if not audio:\n        yield ["กรุณาใส่ไฟล์เพลงก่อน"] + hidden_outputs()\n        return\n    if not model:\n        yield ["กรุณาเลือกโมเดล"] + hidden_outputs()\n        return\n    if CURRENT["proc"] and CURRENT["proc"].poll() is None:\n        yield ["ยังมีงานแยกเสียงทำอยู่ กรุณารอให้เสร็จ หรือกดหยุดก่อน"] + hidden_outputs()\n        return\n\n    song = clean_name(os.path.splitext(os.path.basename(audio))[0])\n    run_dir = os.path.join(OUT_ROOT, song)\n    in_dir = os.path.join(run_dir, "_input")\n    shutil.rmtree(in_dir, ignore_errors=True)\n    os.makedirs(in_dir, exist_ok=True)\n    shutil.copy(audio, os.path.join(in_dir, song + os.path.splitext(audio)[1]))\n    if os.path.exists(PROGRESS_FILE):\n        os.remove(PROGRESS_FILE)\n\n    cmd = [MV_PY, "inference.py", "--target", "vocals", "--exp_name", model, "--model_dir", "checkpoints",\n           "--use_gpu", "y", "--vad_method", "spec", "--spectral_features", "mfcc", "--w2v_ckpt_dir", "pretrained",\n           "--w2v_nth_layer_output", "0", "--use_ema_model", "y", "--mix_consistent_out", "y", "--reorder_chunks", "y",\n           "--skip_error", "n", "--separate_storage", "n", "--output_format", output_format,\n           "--inference_data_dir", in_dir, "--results_save_dir", run_dir]\n    if overlapadd != "None":\n        cmd += ["--use_overlapadd", overlapadd]\n    env = dict(os.environ, MEDLEYVOX_PROGRESS=PROGRESS_FILE, MPLBACKEND="Agg", PYTHONIOENCODING="utf-8")\n    log_path = os.path.join(run_dir, "_log.txt")\n    log = open(log_path, "w", encoding="utf-8")\n    start = time.time()\n    CURRENT["stopped"] = False\n    CURRENT["proc"] = subprocess.Popen(cmd, cwd=MV_DIR, stdout=log, stderr=subprocess.STDOUT, env=env,\n                                       creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0))\n    while CURRENT["proc"].poll() is None:\n        yield [progress_text(read_progress(), start)] + hidden_outputs()\n        time.sleep(1)\n    log.close()\n    CURRENT["proc"] = None\n    p = read_progress() or {}\n    took = fmt_time(time.time() - start)\n    shutil.rmtree(in_dir, ignore_errors=True)\n\n    if CURRENT["stopped"]:\n        yield ["⏹ หยุดการแยกเสียงแล้ว"] + hidden_outputs()\n        return\n    outs = []\n    for i in (1, 2):\n        src = os.path.join(run_dir, f"{song}_output_{i}.{output_format}")\n        dst = os.path.join(run_dir, f"{song}_นักร้อง{i}.{output_format}")\n        if os.path.exists(src):\n            if os.path.exists(dst):\n                os.remove(dst)\n            os.rename(src, dst)\n            outs.append(dst)\n    if p.get("stage") != "finished" or len(outs) != 2:\n        try:\n            with open(log_path, encoding="utf-8", errors="replace") as f:\n                tail = "".join(f.readlines()[-8:])\n        except Exception:\n            tail = ""\n        yield [f"❌ แยกเสียงไม่สำเร็จ\\n{tail}"] + hidden_outputs()\n        return\n    yield ([f"✅ แยกเสียงเสร็จแล้ว ใช้เวลา {took}\\nได้ 2 ไฟล์: นักร้อง 1 และ นักร้อง 2 ฟังเทียบด้านล่าง "\n            "แล้วกด \\"ส่งไปแปลงหลายโมเดล\\" ที่ไฟล์ที่ต้องการได้เลย"]\n           + [gr.update(value=make_preview(outs[0]), visible=True), gr.update(value=make_preview(outs[1]), visible=True)]\n           + [outs[0], outs[1]] + [outs[0], outs[1]] + [gr.update(visible=True)] * 2)\n\n\ndef stop():\n    if CURRENT["proc"] and CURRENT["proc"].poll() is None:\n        CURRENT["stopped"] = True\n        CURRENT["proc"].kill()\n        return "⏹ หยุดการแยกเสียงแล้ว"\n    return "ไม่มีงานแยกเสียงที่กำลังทำอยู่"\n\n\ndef send_to_convert(path):\n    # ส่งไฟล์ไปใส่ช่องเพลงของแท็บ MultiModelConvert (ของจริงไปที่ original, ตัวอย่างไปโชว์)\n    if not path or not os.path.exists(path):\n        gr.Warning("ยังไม่มีไฟล์ให้ส่ง กรุณาแยกเสียงก่อน")\n        return gr.update(), None\n    gr.Info("ส่งไปแล้ว ไปที่แท็บ Plugins > MultiModelConvert แล้วติ๊กโมเดลได้เลย")\n    preview = shared.get("convert_make_preview", make_preview)(path)\n    return gr.update(value=preview), path\n\n\ndef bind_cross_tab():\n    # ผูกปุ่มส่งเข้ากับช่องของ MultiModelConvert เมื่อทั้งสองแท็บสร้างเสร็จแล้ว (ไม่ว่าใครสร้างก่อน)\n    if shared.get("bound") or "convert_audio" not in shared or "mv_send" not in shared:\n        return\n    for btn, state in shared["mv_send"]:\n        btn.click(send_to_convert, [state], [shared["convert_audio"], shared["convert_original"]])\n    shared["bound"] = True\n\n\nshared["bind_cross_tab"] = bind_cross_tab\n\n\ndef applio_plugin():\n    gr.Markdown("## แยกเสียงนักร้อง 2 คน (MedleyVox)\\nใส่เพลงที่มีคนร้อง 2 คน (แยกดนตรีออกแล้ว) → กดแยก → ได้ 2 ไฟล์ → ส่งไปแปลงหลายโมเดลได้ทันที")\n    if not installed():\n        gr.Markdown("⚠️ **ยังไม่ได้ติดตั้ง MedleyVox ในรอบนี้** — เปิดสวิตช์ `install_medleyvox` ด้านบนโน้ตบุ๊ก แล้วรันใหม่")\n    models = list_models()\n    audio = gr.Audio(label="เพลงที่จะแยก (เสียงร้อง 2 คน)", type="filepath")\n    original = gr.State(None)\n    with gr.Row():\n        model = gr.Dropdown(models, value="vocals_238" if "vocals_238" in models else (models[0] if models else None),\n                            label="โมเดล", info=MODEL_HINT, scale=2)\n        overlapadd = gr.Radio(["None", "ola", "ola_norm", "w2v", "w2v_chunk", "sf_chunk"], value="ola",\n                              label="วิธีแบ่งช่วงประมวลผล — แนะนำ ola", info=OLA_HINT, scale=3)\n    output_format = gr.Radio(["wav", "flac", "mp3"], value="wav", label="ชนิดไฟล์ผลลัพธ์")\n    with gr.Row():\n        go = gr.Button("แยกเสียง", variant="primary", scale=3)\n        stop_btn = gr.Button("หยุด", variant="stop", scale=1)\n    status = gr.Textbox(label="สถานะ", lines=4, interactive=False)\n    with gr.Row():\n        with gr.Column():\n            player1 = gr.Audio(label="🎧 นักร้อง 1", visible=False, interactive=False)\n            file1 = gr.File(label="โหลดไฟล์เต็ม นักร้อง 1")\n            send1 = gr.Button("ส่งนักร้อง 1 ไปแปลงหลายโมเดล", visible=False)\n        with gr.Column():\n            player2 = gr.Audio(label="🎧 นักร้อง 2", visible=False, interactive=False)\n            file2 = gr.File(label="โหลดไฟล์เต็ม นักร้อง 2")\n            send2 = gr.Button("ส่งนักร้อง 2 ไปแปลงหลายโมเดล", visible=False)\n    path1, path2 = gr.State(None), gr.State(None)\n\n    audio.upload(on_upload, [audio], [original, audio])\n    audio.clear(lambda: None, None, [original])\n    go.click(separate, [audio, original, model, overlapadd, output_format],\n             [status, player1, player2, file1, file2, path1, path2, send1, send2])\n    stop_btn.click(stop, None, [status])\n\n    shared["mv_send"] = [(send1, path1), (send2, path2)]\n    bind_cross_tab()\n'}
for plugin_name, plugin_code in plugin_sources.items():
    plugin_dir = repo_path / 'tabs' / 'plugins' / 'installed' / plugin_name
    plugin_dir.mkdir(parents=True, exist_ok=True)
    (plugin_dir / '__init__.py').write_text('', encoding='utf-8')
    (plugin_dir / 'plugin.py').write_text(plugin_code, encoding='utf-8')

if tensorboard:
    %load_ext tensorboard
    %tensorboard --logdir logs --bind_all

if sharing_method == 'gradio':
    !python app.py --listen --share --client
elif sharing_method == 'localtunnel':
    !echo Password IP: $(curl --silent https://ipv4.icanhazip.com)
    !lt --port 6969 & python app.py --listen --client
elif sharing_method == 'ngrok':
    import ngrok
    ngrok.kill()
    listener = await ngrok.forward(6969, authtoken=ngrok_token)
    print(f'Ngrok URL: {listener.url()}')
    !python app.py --listen --client
